<a href="https://colab.research.google.com/github/ThiagoPoppe/lamir2026-summer-code/blob/main/fingerprinting_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬️ Audio Fingerprinting Experiments: Analyzing Fingerprint Robustness to Audio Degradation

This notebook explores the robustness of the fingerprinting algorithm against different audio modifications using [GTZAN](https://www.cs.cmu.edu/~gtzan/work/pubs/tsap02gtzan.pdf), a well-known dataset in the field of Music Information Retrieval (MIR).

Although this dataset was not originally designed for audio fingerprinting, but rather for music genre classification, it serves as a useful resource, providing 30-second audio samples from different music genres. This allows us to conduct experiments aimed at investigating the limitations of the fingerprinting algorithm proposed by [Wang em 2003](https://www.ee.columbia.edu/~dpwe/papers/Wang03-shazam.pdf)..

In [1]:
# If running on Google Colab, uncomment and run the following lines.

# !git clone https://github.com/ThiagoPoppe/lamir2026-summer-code.git
# %cd lamir2026-summer-code
# %pip install -q -r requirements.txt

# from google.colab import output
# output.enable_custom_widget_manager()

In [2]:
import numpy as np
import ipywidgets as widgets
import IPython.display as ipd
import matplotlib.pyplot as plt

from audiomentations import (
    Compose,
    PitchShift,
    TimeStretch,
    BandPassFilter,
    ClippingDistortion,
    Mp3Compression,
    AddGaussianSNR
)

from source.utils import load_audio, compute_spectrogram
from source.gtzan import download, build_catalog_dataframe
from source.fingerprint import compute_constellation_map, compute_hashes
from source.config import AudioConfig, FeatureConfig, PeakConfig, HashConfig

from source.visualization.common import play_song
from source.visualization.gtzan import plot_top10_artists_per_genre, song_selector

## 1. The GTZAN Dataset

Criado por [George Tzanetakis (2002)](https://www.cs.cmu.edu/~gtzan/work/pubs/tsap02gtzan.pdf), o conjunto de dados GTZAN é usado amplamente para a avaliação de detecção automática de gênero musical. Por conta disso, a base contém uma distribuição igualitária de 1000 clipes de 30 segundos cada em 10 gêneros musicais.

Na célula abaixo nós carregamos esse dataset e exibimos algumas estatísticas do seu conteúdo.

> É importante notar que por mais que o dataset é extremamente balanceado, há algumas faixas duplicadas e um arquivo em específico de Jazz que resulta em um erro ao carregarmos. Por esse motivo, excluímos esse arquivo, mas deixamos o restante da base de dados intacta. Caso você tenha interesse em uma análise mais aprofundada dessa base de dados, juntamente com suas limitações, críticas, entre outros, é recomendado a leitura do artigo de [Bob L. Sturm (2024)](https://arxiv.org/pdf/1306.1461).



In [3]:
dataset_path, audio_dir = download(output_dir='./gtzan_dataset')

In [4]:
print('Dataset downloaded to:', dataset_path)
print('Audio files are in:', audio_dir)

Dataset downloaded to: ./gtzan_dataset
Audio files are in: ./gtzan_dataset/Data/genres_original


O conjunto de dados do GTZAN não especifica o nome das músicas nem o artista associado. Para isso, iremos fazer um cruzamento dos dados com o [repositório do Bob L. Sturm](https://github.com/boblsturm/GTZAN/tree/main) que contém alguns metadados como quais arquivos devem ser usados para o treinamento e avaliação de modelos e, em particular, no que estamos interessados, um mapeamento de nome do arquivo para o nome da música e artista, respectivamente.

In [5]:
"""Nothing here, change the markdown!"""

'Nothing here, change the markdown!'

Como mencionado anteriormente, o arquivo `jazz.00054.wav` aparenta estar corrimpido, não permitindo assim o seu carregamento. Então iremos remover ele do nosso catálogo e criar um `Dataframe` para deixar todas as informações necessárias organizadas.

In [6]:
catalog_df = build_catalog_dataframe(audio_dir=audio_dir)

In [7]:
print(f"Total valid tracks indexed: {len(catalog_df)}")
catalog_df

Total valid tracks indexed: 999


,track_id,filename,genre,artist,title,display_name,filepath
0,blues.00000,blues.00000.wav,blues,John Lee Hooker,"One Bourbon, One Scotch And One Beer","John Lee Hooker - One Bourbon, One Scotch And ...",./gtzan_dataset/Data/genres_original/blues/blu...
1,blues.00001,blues.00001.wav,blues,John Lee Hooker,I'm In The Mood For Love,John Lee Hooker - I'm In The Mood For Love,./gtzan_dataset/Data/genres_original/blues/blu...
2,blues.00002,blues.00002.wav,blues,John Lee Hooker,Think Twice Before You Go,John Lee Hooker - Think Twice Before You Go,./gtzan_dataset/Data/genres_original/blues/blu...
3,blues.00003,blues.00003.wav,blues,John Lee Hooker,I'm Bad Like Jesse James,John Lee Hooker - I'm Bad Like Jesse James,./gtzan_dataset/Data/genres_original/blues/blu...
4,blues.00004,blues.00004.wav,blues,John Lee Hooker,Walkin' the Boogie,John Lee Hooker - Walkin' the Boogie,./gtzan_dataset/Data/genres_original/blues/blu...
...,...,...,...,...,...,...,...
994,rock.00095,rock.00095.wav,rock,The Stone Roses,Bye Bye Bad Man,The Stone Roses - Bye Bye Bad Man,./gtzan_dataset/Data/genres_original/rock/rock...
995,rock.00096,rock.00096.wav,rock,The Stone Roses,(Song For My) Sugar Spun Sister,The Stone Roses - (Song For My) Sugar Spun Sister,./gtzan_dataset/Data/genres_original/rock/rock...
996,rock.00097,rock.00097.wav,rock,The Stone Roses,Made Of Stone,The Stone Roses - Made Of Stone,./gtzan_dataset/Data/genres_original/rock/rock...
997,rock.00098,rock.00098.wav,rock,The Stone Roses,Shoot You Down,The Stone Roses - Shoot You Down,./gtzan_dataset/Data/genres_original/rock/rock...


Com isso, podemos fazer estatísticas básicas, como por exemplo verificar quais são os top 10 artistas que mais aparecem no catálogo de trechos de músicas.

In [8]:
possible_genres = ['All'] + sorted(catalog_df.genre.unique())

widgets.interact(
    plot_top10_artists_per_genre,
    catalog_df=widgets.fixed(catalog_df),
    genre=widgets.Dropdown(options=possible_genres, description='Genre')   
)

interactive(children=(Dropdown(description='Genre', options=('All', 'blues', 'classical', 'country', 'disco', …

<function source.visualization.gtzan.plot_top10_artists_per_genre(catalog_df, genre)>

Nós também podemos filtrar o catálogo por um artista específico, por exemplo `Queen` e carregar um clipe de áudio da música `Bohemian Rhapsody` por exemplo.

In [9]:
song_selector(catalog_df=catalog_df)

Agora que nós introduzimos a base de dados que iremos utilizar, vamos para os experimentos.

## 2. Experiments

### 2.1. Configuration classes and useful functions

To focus this section only on the experiments itself, we provide a collection of useful functions that will help you guide your own experiments and take your own conclusions. Please, read the documentation of each function, as you will be asked to modify some of its parameters along this section.

### 2.1. Fingerprint Accuracy with Query Size

### 2.2. Fingerprint Robustness to Spectrogram Parameters

### 2.3. Fingerprint Robustness to Audio Degradation

So far, we have only extracted the fingerprint of a pristine, well-behaved audio track. However, real-world queries, like the ones received by Shazam, are rarely perfect. This was especially true back in 2003 when the paper was published. At that time, the algorithm had to accurately identify songs recorded in noisy bars, captured through primitive flip-phone microphones, and heavily compressed over 2G cellular networks.

To test the robustness of our constellation maps, we will artificially degrade our audio query to simulate an early 2000s mobile phone recording. We will use the [audiomentations](https://github.com/iver56/audiomentations) Python library, which makes it easy to apply realistic audio distortions.

In [ ]:
zearly_mobile_pipeline = Compose([
    # 1. Simulate the narrow frequency band of old 2G cellular networks
    BandPassFilter(min_center_freq=1500.0, max_center_freq=2000.0, min_bandwidth_fraction=1.0, max_bandwidth_fraction=1.5, p=1.0),

    # 2. Apply heavy digital compression (early GSM networks were ~13 kbps)
    Mp3Compression(min_bitrate=8, max_bitrate=32, backend="fast-mp3-augment", p=1.0),

    # 3. Simulate a cheap microphone clipping
    ClippingDistortion(min_percentile_threshold=1, max_percentile_threshold=5, p=1.0),

    # 4. Add proportional static hiss (using SNR ensures it scales with track volume)
    AddGaussianSNR(min_snr_db=10.0, max_snr_db=15.0, p=1.0)
])

In [ ]:
print('Original Waveform')
ipd.display(ipd.Audio(data=wave, rate=sr))

print('Artificially Degraded Waveform')
degraded_wave = early_mobile_pipeline(samples=wave, sample_rate=sr)
ipd.display(ipd.Audio(data=degraded_wave, rate=sr))

In [ ]:
degraded_stft = np.abs(librosa.stft(degraded_wave))[:128, :]
degraded_constellation_map = create_constellation_map(degraded_stft)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Constellation Map Robustness: Original vs. Artifically Degraded Recording')

img = ax.imshow(np.log1p(filtered_stft_mag), cmap='magma', origin='lower', aspect='auto')

original_peaks = np.where(constellation_map)
ax.scatter(original_peaks[1], original_peaks[0], color='red', marker='o', s=15, label='Original Peaks')

degraded_peaks = np.where(degraded_constellation_map)
ax.scatter(degraded_peaks[1], degraded_peaks[0], color='lightblue', marker='o', s=8, label='Degraded Peaks')

ax.set_xlabel('Spectrogram Frames')
ax.set_ylabel('Frequency Bins')
fig.legend(loc='upper right')
fig.tight_layout()
plt.show()

Looking at the plot above, you will notice that while some of the weaker peaks are swallowed by the noise and compression, the core "skeleton" of our constellation map survives intact. This highlights the genius of the algorithm: because it ultimately relies on the relative positioning of these points rather than their exact amplitudes, capturing just a subset of the original peaks is more than enough to yield a match.

### 2.4. Fingerprint Robustness to Audio Structure

However, noise and compression only alter the *energy* of the audio. What happens when a distortion alters the structural *coordinates* themselves, like if a DJ speeds up the track or shifts the pitch? Let's build a new pipeline to test the algorithm's limits.

In [ ]:
remix_pipeline = Compose([
    # 1. Speed up the track by 10% (alters the time coordinates)
    TimeStretch(min_rate=1.10, max_rate=1.10, p=1.0),

    # 2. Drop the pitch by exactly one semitone (alters the frequency coordinates)
    PitchShift(min_semitones=-1, max_semitones=-1, p=1.0)
])

In [ ]:
print('Original Waveform')
ipd.display(ipd.Audio(data=wave, rate=sr))

print('Remixed Pitch Waveform')
remixed_wave = remix_pipeline(samples=wave, sample_rate=sr)
ipd.display(ipd.Audio(data=remixed_wave, rate=sr))

In [ ]:
remixed_stft = np.abs(librosa.stft(remixed_wave))[:128, :]
remixed_constellation_map = create_constellation_map(remixed_stft)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Constellation Map Robustness: Original vs. Remixed Recording')

img = ax.imshow(np.log1p(filtered_stft_mag), cmap='magma', origin='lower', aspect='auto')

original_peaks = np.where(constellation_map)
ax.scatter(original_peaks[1], original_peaks[0], color='red', marker='o', s=15, label='Original Peaks')

remixed_peaks = np.where(remixed_constellation_map)
ax.scatter(remixed_peaks[1], remixed_peaks[0], color='lightblue', marker='o', s=8, label='Remixed Peaks')

ax.set_xlabel('Spectrogram Frames')
ax.set_ylabel('Frequency Bins')
fig.legend(loc='upper right')
fig.tight_layout()
plt.show()

As you can see, almost no peaks survived our "remixed" pipeline. This reveals the primary weakness of this type of retrieval algorithm: it is designed strictly for *exact-audio matching*. It cannot handle tempo changes, pitch shifts, or cover versions.

However, you can still retrieve these modified tracks using alignment algorithms like [Dynamic Time Warping (DTW)](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C7/C7S2_SubsequenceDTW.html), which are computationally much heavier than the fingerprinting approach explained here. If you want to explore this area further, the [Fundamentals of Music Processing (FMP) Notebooks](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C7/C7S2_AudioMatching.html) by *Meinard Müller* are highly recommended.

## 3. Discussions